In [10]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [ ]:
ngrok_ip = "0.tcp.in.ngrok.io:10122"

In [7]:
topics = {"sales": "sales_topic",
          "employees": "employees_topic",
          "expenses": "expenses_topic",
          "regions": "regions_topic"}

checkpoints = {"sales": "abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/dev_checkpoints/brz_checkpoints/sales_checkpoints",
               "employees": "abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/dev_checkpoints/brz_checkpoints/employees_checkpoints",
               "expenses": "abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/dev_checkpoints/brz_checkpoints/expenses_checkpoints",
               "regions": "abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/dev_checkpoints/brz_checkpoints/regions_checkpoints"
               }

In [15]:
from pyspark.sql import functions as F

def consume_topic(topic_key):

      df = (spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", ngrok_ip)
            .option("subscribe", topics[topic_key])
            .option("startingOffsets", "earliest")
            .option("failOnDataLoss", "false")
            .load()
            )

      df = df.selectExpr("cast(key as string) as key",
                        "cast(value as string) as value",
                        "topic",
                        "partition",
                        "offset",
                        "timestamp").withColumn("processed_time", F.current_timestamp())

      query = (df.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoints[topic_key])
            .trigger(availableNow = True)
            .outputMode("append")
            .table(f"sales_streaming_dev.brz.{topic_key}_raw")
            )

      return query

queries = {}

for topic_key, value in topics.items():

      q = consume_topic(topic_key)
      queries[topic_key] = q

for topic_key, q in queries.items():

      try:
            q.awaitTermination(60)
            print(f"Completed stream for {topic_key}")
      
      except Exception as e:

            raise(f"error in consuming {topic_key}: {e}")

Completed stream for sales
Completed stream for employees
Completed stream for expenses
Completed stream for regions


In [9]:
spark.sql("select * from sales_streaming_dev.brz.sales_raw limit 10")

,key,value,topic,partition,offset,timestamp,processed_time
